<a href="https://colab.research.google.com/github/nonmaclo/signate-practice/blob/main/J_League_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### 課題内容
Jリーグ公式戦2014年シーズン後半戦全試合の観客動員数を予測するモデルを作成していただきます。


### 評価方法
精度評価は、評価関数「RMSE（Root Mean Squared Error 平均二乗平方根誤差）」を使用します。
評価値は0以上の値をとり、精度が高いほど小さな値となります。
暫定評価は、評価用データの一部に対する評価結果になります。
$$
\mathrm{RMSE}
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}
(y_i-\hat{y}_i)^2
}
$$

### それぞれの意味
- \(N\)：データの数
- \(y_i\)：実際の値
- \(\hat{y}_i\)：モデルの予測値
- \(y_i-\hat{y}_i\)：予測の誤差
- \((y_i-\hat{y}_i)^2\)：誤差を二乗
- \(\sum\)：全部のデータについて足す
- \(\frac{1}{N}\)：平均を取る
- \(\sqrt{}\)：最後に平方根を取る


### ルール
ある日の予測をする時は、その日に確定している情報のみ使用可

データ数 ： 1,722行 データ説明 ： 対戦カードとその対戦の観客数等を記したデータ

| カラム | ヘッダ名称 | データ型 | 説明 |
|---:|---|---|---|
| 0 | id | int | 対戦カードID |
| 1 | y | int | 観客数（目的変数） |
| 2 | year | int | 開催年度 |
| 3 | stage | varchar | 開催大会 |
| 4 | match | varchar | 開催節 |
| 5 | gameday | varchar | 試合日 |
| 6 | time | varchar | キックオフ時刻 |
| 7 | home | varchar | ホームチーム |
| 8 | away | varchar | アウェイチーム |
| 9 | stadium | varchar | スタジアム |
| 10 | tv | varchar | TV放送 |

### condition
- スタジアム・試合環境：weather、temperature、humidity
- 審判：referee
- チーム・スタメン：home_team、home_01〜、away_01〜
試合結果：home_score、away_score

天候による変化や特定のスター選手のありなしの変化も終えれば追いたい、例えばスコアは試合後に確定する情報なので、予測時点で利用できないなら特徴量には使えない。
一方、スタジアム、天候、気温、湿度、ホーム・アウェイチーム、スタメンなどは、データの作られ方と予測条件を確認したうえで利用できる可能性がある。

| id | home_score | away_score | weather | temperature | humidity | referee | home_team | home_01 | home_02 | ... | away_02 | away_03 | away_04 | away_05 | away_06 | away_07 | away_08 | away_09 | away_10 | away_11 |
|---:|---:|---:|---|---:|---:|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 13994 | 1 | 0 | 雨 | 3.8 | 66% | 木村　博之 | ベガルタ仙台 | 林　卓人 | 菅井　直樹 | ... | 新井場　徹 | 岩政　大樹 | 中田　浩二 | アレックス | 青木　剛 | 増田　誓志 | 小笠原　満男 | 本山　雅志 | 大迫　勇也 | ジュニーニョ |
| 1 | 13995 | 1 | 0 | 屋内 | 12.4 | 43% | 西村　雄一 | 名古屋グランパス | 楢﨑　正剛 | 田中　隼磨 | ... | 吉田　豊 | 岩下　敬輔 | カルフィン　ヨン　ア　ピン | 李　記帝 | 村松　大輔 | 河井　陽介 | 枝村　匠馬 | 高木　俊幸 | アレックス | 大前　元紀 |
| 2 | 13996 | 2 | 3 | 晴一時雨 | 11.3 | 41% | 高山　啓義 | ガンバ大阪 | 藤ヶ谷　陽介 | 加地　亮 | ... | 近藤　岳登 | 北本　久仁衛 | 伊野波　雅彦 | 相馬　崇人 | 三原　雅俊 | 田中　英雄 | 野沢　拓也 | 橋本　英郎 | 森岡　亮太 | 大久保　嘉人 |

### stadium
name,address,capaが記入されているので最低でもまずはキャパによる推定材料として使用できる
| name | address | capa |
|---|---|---:|
| 名古屋市瑞穂陸上競技場 | 愛知県名古屋市瑞穂区山下通5-1 | 20000 |
| 豊田スタジアム | 愛知県豊田市千石町7-2 | 40000 |
| フクダ電子アリーナ | 千葉県千葉市中央区川崎町1-20 | 18500 |
| 日立柏サッカー場 | 千葉県柏市日立台1-2-50 | 15349 |
| ニンジニアスタジアム | 愛媛県松山市上野町乙46 | 15576 |


In [8]:
import pandas as pd
train = pd.read_csv("/content/drive/MyDrive/jleague/train.csv")
test = pd.read_csv("/content/drive/MyDrive/jleague/test.csv")
train_add = pd.read_csv("/content/drive/MyDrive/jleague/train_add.csv")
stadium = pd.read_csv("/content/drive/MyDrive/jleague/stadium.csv")
sample_submit = pd.read_csv("/content/drive/MyDrive/jleague/sample_submit.csv")
condition_add = pd.read_csv("/content/drive/MyDrive/jleague/condition_add.csv")
condition = pd.read_csv("/content/drive/MyDrive/jleague/condition.csv")
add_2014 = pd.read_csv("/content/drive/MyDrive/jleague/2014_add.csv")

condition.head()

,id,home_score,away_score,weather,temperature,humidity,referee,home_team,home_01,home_02,...,away_02,away_03,away_04,away_05,away_06,away_07,away_08,away_09,away_10,away_11
0,13994,1,0,雨,3.8,66%,木村 博之,ベガルタ仙台,林 卓人,菅井 直樹,...,新井場 徹,岩政 大樹,中田 浩二,アレックス,青木 剛,増田 誓志,小笠原 満男,本山 雅志,大迫 勇也,ジュニーニョ
1,13995,1,0,屋内,12.4,43%,西村 雄一,名古屋グランパス,楢﨑 正剛,田中 隼磨,...,吉田 豊,岩下 敬輔,カルフィン ヨン ア ピン,李 記帝,村松 大輔,河井 陽介,枝村 匠馬,高木 俊幸,アレックス,大前 元紀
2,13996,2,3,晴一時雨,11.3,41%,高山 啓義,ガンバ大阪,藤ヶ谷 陽介,加地 亮,...,近藤 岳登,北本 久仁衛,伊野波 雅彦,相馬 崇人,三原 雅俊,田中 英雄,野沢 拓也,橋本 英郎,森岡 亮太,大久保 嘉人
3,13997,1,0,曇一時雨のち晴,11.4,52%,松尾 一,サンフレッチェ広島,西川 周作,森脇 良太,...,濱田 水輝,阿部 勇樹,槙野 智章,平川 忠亮,鈴木 啓太,山田 直輝,梅崎 司,柏木 陽介,原口 元気,田中 達也
4,13998,0,0,屋内,22.5,32%,廣瀬 格,コンサドーレ札幌,李 昊乗,高木 純平,...,駒野 友一,チョ ビョングク,藤田 義明,山本 脩斗,小林 裕紀,山本 康裕,山田 大記,松浦 拓弥,菅沼 実,前田 遼一


In [9]:
print('train shape: ', train.shape)
print('train_add shape: ', train_add.shape)
print('test shape: ',test.shape)
print('condition shape: ', condition.shape)
print('condition_add shape: ', condition_add.shape)
print('stadium shape:', stadium.shape)

train shape:  (1721, 11)
train_add shape:  (232, 11)
test shape:  (313, 10)
condition shape:  (2034, 31)
condition_add shape:  (270, 31)
stadium shape: (59, 3)


In [10]:
full_train = pd.concat([train,train_add],axis=0)
full_condition = pd.concat([condition,condition_add],axis=0)

In [11]:
print('train concat')
print('before: ', train.shape, train_add.shape)
print('after: ', full_train.shape)

train concat
before:  (1721, 11) (232, 11)
after:  (1953, 11)


また、full_trainとfull_conditionはidを、full_trainとstadiumはスタジアム名を参照して結合できそうです。

今回は選手、レフェリーは使用せず分析に使いやすそうな特徴量のみを結合していきます。

train, testのデータセットを仕上げていきましょう。

In [13]:
# 結合する特徴量を選択
# stadiumdfの列名nameをstadiumに変更
stadium = stadium.rename(columns={'name': 'stadium'})
# 必要な列だけ残す
stadium = stadium[['stadium', 'capa']]
# 必要な列だけ残す
full_condition = full_condition[['id' ,'weather' ,'temperature' ,'humidity']]

In [14]:
# 結合
full_train = pd.merge(full_train, stadium, on='stadium',  how='left')
full_train = pd.merge(full_train, full_condition, on='id',  how='left')
full_test = pd.merge(test, stadium, on='stadium',  how='left')
full_test = pd.merge(full_test, full_condition, on='id',  how='left')

### how='left' は？
左側のDataFrame（この場合 full_train）を基準にするという意味。full_train の行を基本的に残したまま、キーが一致する情報を右側から追加する

In [15]:
# 結合後の確認
display(full_train.head(), full_test.head())
print('full_train shape: ', full_train.shape)
print('full_test shape: ', full_test.shape)

,id,y,year,stage,match,gameday,time,home,away,stadium,tv,capa,weather,temperature,humidity
0,13994,18250,2012,Ｊ１,第１節第１日,03/10(土),14:04,ベガルタ仙台,鹿島アントラーズ,ユアテックスタジアム仙台,スカパー／ｅ２／スカパー光／ＮＨＫ総合,19694,雨,3.8,66%
1,13995,24316,2012,Ｊ１,第１節第１日,03/10(土),14:04,名古屋グランパス,清水エスパルス,豊田スタジアム,スカパー／ｅ２／スカパー光（Ｊ ＳＰＯＲＴＳ ４）／ＮＨＫ名古屋,40000,屋内,12.4,43%
2,13996,17066,2012,Ｊ１,第１節第１日,03/10(土),14:04,ガンバ大阪,ヴィッセル神戸,万博記念競技場,スカパー／ｅ２／スカパー光（Ｊ ＳＰＯＲＴＳ １）／ＮＨＫ大阪,21000,晴一時雨,11.3,41%
3,13997,29603,2012,Ｊ１,第１節第１日,03/10(土),14:06,サンフレッチェ広島,浦和レッズ,エディオンスタジアム広島,スカパー／ｅ２／スカパー光／ＮＨＫ広島,50000,曇一時雨のち晴,11.4,52%
4,13998,25353,2012,Ｊ１,第１節第１日,03/10(土),14:04,コンサドーレ札幌,ジュビロ磐田,札幌ドーム,スカパー／ｅ２／スカパー光（スカイ・Ａ ｓｐｏｒｔｓ＋）／ＮＨＫ札幌,39232,屋内,22.5,32%


,id,year,stage,match,gameday,time,home,away,stadium,tv,capa,weather,temperature,humidity
0,15822,2014,Ｊ１,第１８節第１日,08/02(土),19:04,ベガルタ仙台,大宮アルディージャ,ユアテックスタジアム仙台,スカパー！／スカパー！プレミアムサービス,19694,晴,27.4,70%
1,15823,2014,Ｊ１,第１８節第１日,08/02(土),18:34,鹿島アントラーズ,サンフレッチェ広島,県立カシマサッカースタジアム,スカパー！／スカパー！プレミアムサービス,40728,晴,30.8,65%
2,15824,2014,Ｊ１,第１８節第１日,08/02(土),19:04,浦和レッズ,ヴィッセル神戸,埼玉スタジアム２００２,スカパー！／スカパー！プレミアムサービス／ＮＨＫ ＢＳ１／テレ玉,63700,晴,31.7,58%
3,15825,2014,Ｊ１,第１８節第１日,08/02(土),19:03,柏レイソル,川崎フロンターレ,日立柏サッカー場,スカパー！／スカパー！プレミアムサービス,15349,晴,29.3,76%
4,15827,2014,Ｊ１,第１８節第１日,08/02(土),19:03,アルビレックス新潟,セレッソ大阪,デンカビッグスワンスタジアム,スカパー！／スカパー！プレミアムサービス,42300,晴,30.4,68%


full_train shape:  (1953, 15)
full_test shape:  (313, 14)


In [16]:
# 欠損値などの確認
full_train.isnull().sum()

,0
id,0
y,0
year,0
stage,0
match,0
gameday,0
time,0
home,0
away,0
stadium,0


In [17]:
full_test.isnull().sum()

,0
id,0
year,0
stage,0
match,0
gameday,0
time,0
home,0
away,0
stadium,0
tv,0


In [18]:
full_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1953 entries, 0 to 1952
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           1953 non-null   int64  
 1   y            1953 non-null   int64  
 2   year         1953 non-null   int64  
 3   stage        1953 non-null   object 
 4   match        1953 non-null   object 
 5   gameday      1953 non-null   object 
 6   time         1953 non-null   object 
 7   home         1953 non-null   object 
 8   away         1953 non-null   object 
 9   stadium      1953 non-null   object 
 10  tv           1953 non-null   object 
 11  capa         1953 non-null   int64  
 12  weather      1953 non-null   object 
 13  temperature  1953 non-null   float64
 14  humidity     1953 non-null   object 
dtypes: float64(1), int64(4), object(10)
memory usage: 229.0+ KB


In [19]:
full_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 313 entries, 0 to 312
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           313 non-null    int64  
 1   year         313 non-null    int64  
 2   stage        313 non-null    object 
 3   match        313 non-null    object 
 4   gameday      313 non-null    object 
 5   time         313 non-null    object 
 6   home         313 non-null    object 
 7   away         313 non-null    object 
 8   stadium      313 non-null    object 
 9   tv           313 non-null    object 
 10  capa         313 non-null    int64  
 11  weather      313 non-null    object 
 12  temperature  313 non-null    float64
 13  humidity     313 non-null    object 
dtypes: float64(1), int64(3), object(10)
memory usage: 34.4+ KB
